In this notebook we are implementing a version of the VAE where we only use the features that have a high variance. This way we reduce the number of fetatures that would otherwise be too large.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
%config Completer.use_jedi = False
from sklearn.feature_selection import VarianceThreshold

In [ ]:
## PREPROCESSING

log_counts = pd.read_csv('log_corrected_counts_trauma.tsv', sep = '\t', index_col = 0)
clin = pd.read_csv('/Users/smasarone/Documents/jupyter_notebooks_copy2/clin_rnaseq.csv', header = 0, index_col = 0)
log_counts = log_counts.T
log_counts = log_counts.filter(items = clin.index, axis = 0)

# import also proteomics
prot = pd.read_csv('/Users/smasarone/Documents/jupyter_notebooks_copy2/raw_data_proteins_filtered.csv',header = 0, index_col = 0)
prot_list = pd.read_csv('/Users/smasarone/Documents/jupyter_notebooks_copy2/prot_filtered.csv', header = 0, index_col=0)
prot = prot.filter(prot_list.iloc[:,0], axis = 1)
prot = prot.drop(labels = [1578], axis = 0)

new_idx_p = []
for i in prot.index:
    new_idx_p.append('S'+str(i))
prot.index = new_idx_p

# filter clin and counts by prot
counts = log_counts.filter(items = prot.index, axis = 0)
clin = clin.filter(items = prot.index, axis = 0)
prot = prot.filter(items = clin.index, axis =0)
print(clin.shape)
print(counts.shape)
print(prot.shape)

clin_selected = pd.DataFrame(clin['age'])
clin_selected['plt'] = clin['baseline_plt']
clin_selected['wcc'] = clin['baseline_wcc']
clin_selected['sbp'] = clin['admission_sbp']
clin_selected['bd'] = clin['baseline_base_deficit']
clin_selected = clin_selected.fillna(value = 0)

all_df = counts.join(clin_selected.join(prot))

In [ ]:
selector = VarianceThreshold(threshold = 5.8)
res = selector.fit_transform(all_df)
res.shape

In [ ]:
mask = selector.get_support()        # to get the right idx
final_col = all_df.columns[mask] # final idx
df_all_filtered = res

In [ ]:
df_to_use = pd.DataFrame(df_all_filtered)
df_to_use.index = all_df.index
df_to_use.columns = final_col

In [ ]:
# name is misleading because scaled counts is actually the merged dataset
scaled_data = StandardScaler().fit_transform(df_to_use)
scaled_data = pd.DataFrame(scaled_data, index = df_to_use.index)
scaled_data.columns = final_col

In [ ]:
import os
import random
from keras import backend as K
seed_value= 0
os.environ['PYTHONHASHSEED']=str(seed_value)
random.seed(seed_value)
np.random.seed(seed_value)
tf.random.set_seed(seed_value)
session_conf = tf.compat.v1.ConfigProto(intra_op_parallelism_threads=1, inter_op_parallelism_threads=1)
sess = tf.compat.v1.Session(graph=tf.compat.v1.get_default_graph(), config=session_conf)
K.set_session(sess)



class Sampling(layers.Layer):
    """Uses (z_mean, z_log_var) to sample z, 
       the vector encoding a digit."""

    def call(self, inputs):
        z_mean, z_log_var = inputs
        batch = tf.shape(z_mean)[0]
        dim = tf.shape(z_mean)[1]
        epsilon = tf.keras.backend.random_normal(shape=(batch, dim))
        return z_mean + tf.exp(0.5 * z_log_var) * epsilon


# ENCODER 
INPUT_SHAPE = scaled_data.shape[1]
latent_dim=12  #4
encoder_inputs = layers.Input(shape=(INPUT_SHAPE), name="x")
h1 = layers.Dense(2800, activation="relu")(encoder_inputs)
h2 = layers.Dense(1000, activation="relu")(h1)
h3 = layers.Dense(300, activation="relu")(h2)
z_mean = layers.Dense(latent_dim, name="z_mean")(h3)
z_log_var = layers.Dense(latent_dim, name="z_log_var")(h3)
z = Sampling()([z_mean, z_log_var])

# summary encoder
encoder = keras.Model(encoder_inputs, [z_mean, z_log_var, z], name="encoder")
encoder.summary()

# DECODER
latent_inputs = keras.Input(shape=(latent_dim))
dec_h1 = layers.Dense(300, activation="relu")(latent_inputs)
dec_h2 = layers.Dense(1000, activation="relu")(dec_h1)
dec_h3 = layers.Dense(2800, activation="relu")(dec_h2)
decoder_outputs = layers.Dense(INPUT_SHAPE, activation="linear")(dec_h3)

# summary decoder
decoder = keras.Model(latent_inputs, decoder_outputs, name="decoder")
decoder.summary()


class VAE(keras.Model):
    def __init__(self, encoder, decoder, **kwargs):
        super(VAE, self).__init__(**kwargs)
        self.encoder = encoder
        self.decoder = decoder
        self.total_loss_tracker = keras.metrics.Mean(name="total_loss")
        self.reconstruction_loss_tracker = keras.metrics.Mean(
            name="reconstruction_loss")
        self.kl_loss_tracker = keras.metrics.Mean(name="kl_loss")

    @property
    def metrics(self):
        return [
            self.total_loss_tracker,
            self.reconstruction_loss_tracker,
            self.kl_loss_tracker,
        ]

    def train_step(self, data):
        with tf.GradientTape() as tape:
            z_mean, z_log_var, z = self.encoder(data)
            reconstruction = self.decoder(z)
            reconstruction_loss = tf.reduce_mean(
                tf.reduce_sum(
                    keras.losses.mse(data, reconstruction)
                )
            )
            kl_loss = -0.5 * (1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var))
            kl_loss = tf.reduce_mean(tf.reduce_sum(kl_loss, axis=1))
            total_loss = reconstruction_loss + kl_loss
        
        grads = tape.gradient(total_loss, self.trainable_weights)
        self.optimizer.apply_gradients(zip(grads, self.trainable_weights))
        self.total_loss_tracker.update_state(total_loss)
        self.reconstruction_loss_tracker.update_state(reconstruction_loss)
        self.kl_loss_tracker.update_state(kl_loss)
        
        return {
            
            "loss": self.total_loss_tracker.result(),
            "reconstruction_loss": self.reconstruction_loss_tracker.result(),
            "kl_loss": self.kl_loss_tracker.result(),
        }
    
vae = VAE(encoder, decoder)
vae.compile(optimizer=keras.optimizers.Adam())
history = vae.fit(scaled_data, epochs=100, batch_size=64)

In [ ]:
plt.title("Train and val loss")
plt.plot(history.history['loss'], label = 'loss')
plt.plot(history.history['reconstruction_loss'], label= 'reconstruction loss')
plt.plot(history.history['kl_loss'],label= 'kl loss')
plt.legend()
plt.show() 

print(np.min(history.history['loss']))

In [ ]:
# get the embedding
get_z = keras.models.Model(inputs=[encoder_inputs], outputs=vae.get_layer("encoder").output, name="VAE")
z_output = get_z.predict({"x": scaled_data})

In [ ]:
# extract only the releavant output
embedding = z_output[2]
pca = PCA(n_components = 4)
pca_res = pca.fit_transform(embedding)

pca_raw =  PCA(n_components = 5)
pca_res_r = pca_raw.fit_transform(scaled_data)

y = clin.iss
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12,4))
fig.suptitle("PCA of input data vs embedding")
ax1.set_title("PCA of expression data")
sns.scatterplot(x = pca_res_r[:,0], y = pca_res_r[:,1], hue=y, palette = 'coolwarm', ax = ax1)
ax2.set_title("PCA of embedding")
sns.scatterplot(x = pca_res[:,0], y = pca_res[:,1], hue=y, palette = 'coolwarm', ax = ax2)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn import model_selection
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix

y = [1 if i >=25 else 0 for i in clin.iss]
X_train, X_test, y_train, y_test = train_test_split(embedding, y, random_state=122, stratify = y)

#try a random forest
rf = RandomForestClassifier(max_depth=2, class_weight = 'balanced', random_state=123)
scoring = ['accuracy', 'precision_weighted', 'recall', 'f1', 'roc_auc']
kfold = model_selection.KFold(n_splits=5, shuffle=True, random_state=125)
cv_results = model_selection.cross_validate(rf, X_train, y_train, cv=kfold, scoring=scoring)

cv_results

In [ ]:
np.mean([0.75294118, 0.62178703, 0.73333333, 0.59180036, 0.85833333])

In [ ]:
from sklearn.linear_model import LogisticRegression
lr = LogisticRegression(class_weight = 'balanced', random_state=123)
scoring = ['accuracy', 'precision_weighted', 'recall', 'f1', 'roc_auc']
kfold = model_selection.KFold(n_splits=5, shuffle=True, random_state=125)
cv_results = model_selection.cross_validate(lr, X_train, y_train, cv=kfold, scoring=scoring)

cv_results

In [ ]:
np.mean([0.81437908, 0.56793146, 0.71372549, 0.62566845, 0.725  ])